In [ ]:
-- =============================================================================
-- SLEEPER TRADES PIPELINE
-- Dynasty-Aware Multi-Season Trade Analysis
-- 
-- Key Features:
-- - Multi-season player point tracking
-- - Draft pick career point tracking
-- - Trade completeness detection (all picks realized)
-- - Dynasty vs redraft league support
-- - Trade impact calculation across multiple time horizons
-- =============================================================================

In [ ]:
-- ---------- PLAYER ASSETS ----------

In [ ]:
-- Player assets in trades (who got which players)
CREATE OR REPLACE MATERIALIZED VIEW fact_trade_player_assets AS
WITH tx AS (SELECT * FROM stg_trade_transactions),
adds_raw AS (
  SELECT
    league_id,
    season,
    transaction_id,
    week,
    event_ts,
    cluster_key,
    cluster_name,
    explode(transform(map_entries(adds), e -> 
      named_struct('player_id', e.key, 'roster_id', e.value)
    )) AS add_struct
  FROM tx
),
adds AS (
  SELECT
    league_id,
    season,
    transaction_id,
    week,
    event_ts,
    cluster_key,
    cluster_name,
    add_struct.player_id,
    add_struct.roster_id AS side_roster_id,
    'incoming' AS direction
  FROM adds_raw
),
drops_raw AS (
  SELECT
    league_id,
    season,
    transaction_id,
    week,
    event_ts,
    cluster_key,
    cluster_name,
    explode(transform(map_entries(drops), e -> 
      named_struct('player_id', e.key, 'roster_id', e.value)
    )) AS drop_struct
  FROM tx
),
drops AS (
  SELECT
    league_id,
    season,
    transaction_id,
    week,
    event_ts,
    cluster_key,
    cluster_name,
    drop_struct.player_id,
    drop_struct.roster_id AS side_roster_id,
    'outgoing' AS direction
  FROM drops_raw
)
SELECT * FROM adds
UNION ALL
SELECT * FROM drops;

In [ ]:
-- Multi-season player points and VOR after trade
-- Uses cluster_key for cross-season tracking (league_id changes each season in Sleeper)
-- Counts ALL career points scored by the player regardless of subsequent ownership
CREATE OR REPLACE MATERIALIZED VIEW fact_trade_player_points_multi_season AS
WITH incoming_players AS (
  SELECT
    league_id,
    transaction_id,
    side_roster_id,
    player_id,
    season AS trade_season,
    week AS trade_week,
    cluster_key
  FROM fact_trade_player_assets
  WHERE direction = 'incoming'
),
-- Get all points and VOR scored by acquired players after the trade
points_after_trade AS (
  SELECT
    ip.league_id,
    ip.transaction_id,
    ip.side_roster_id,
    ip.player_id,
    ip.trade_season,
    ip.trade_week,
    pw.season AS scoring_season,
    pw.week AS scoring_week,
    pw.points,
    COALESCE(vor.vor, 0.0) AS vor,
    -- Calculate seasons since trade
    CAST(pw.season AS INT) - CAST(ip.trade_season AS INT) AS seasons_after_trade,
    -- Calculate weeks since trade (approximate, doesn't account for season boundaries)
    CASE
      WHEN pw.season = ip.trade_season THEN pw.week - ip.trade_week
      ELSE NULL  -- Cross-season week calculations are complex, use season delta instead
    END AS weeks_after_trade
  FROM incoming_players ip
  JOIN workspace.sleeper_core.fact_player_week_enriched pw
    ON ip.cluster_key = pw.cluster_key
    AND ip.player_id = pw.player_id
  LEFT JOIN workspace.sleeper_core.agg_player_week_vor vor
    ON pw.league_id = vor.league_id
    AND pw.season = vor.season
    AND pw.week = vor.week
    AND pw.player_id = vor.player_id
  WHERE
    -- Only count points after the trade
    (CAST(pw.season AS INT) > CAST(ip.trade_season AS INT))
    OR (pw.season = ip.trade_season AND pw.week > ip.trade_week)
)
SELECT
  league_id,
  transaction_id,
  side_roster_id,
  player_id,
  trade_season,
  trade_week,
  scoring_season,
  scoring_week,
  points,
  vor,
  seasons_after_trade,
  weeks_after_trade
FROM points_after_trade;

In [ ]:
-- ---------- DRAFT PICK ASSETS ----------

In [ ]:
-- Draft picks included in trades with realization tracking
-- Key insight: roster_id in traded picks = ORIGINAL owner of the pick (whose pick it naturally is)
CREATE OR REPLACE MATERIALIZED VIEW fact_trade_pick_assets AS
WITH tx AS (
  SELECT league_id, transaction_id, season AS trade_season, draft_picks
  FROM stg_trade_transactions
),
parsed AS (
  SELECT 
    league_id,
    transaction_id,
    trade_season,
    transform(
      CAST(draft_picks AS ARRAY<STRING>),
      p -> from_json(p, 'MAP<STRING,STRING>')
    ) AS picks
  FROM tx
  WHERE draft_picks IS NOT NULL AND size(CAST(draft_picks AS ARRAY<STRING>)) > 0
),
exploded AS (
  SELECT 
    league_id,
    transaction_id,
    trade_season,
    explode(picks) AS pick_map
  FROM parsed
),
normalized AS (
  SELECT
    league_id,
    transaction_id,
    trade_season,
    CAST(pick_map['round'] AS INT) AS round,
    CAST(pick_map['roster_id'] AS INT) AS original_owner_id,  -- This is the ORIGINAL owner
    CAST(pick_map['previous_owner_id'] AS INT) AS from_roster_id,  -- Who's trading it away
    CAST(pick_map['owner_id'] AS INT) AS to_roster_id,  -- Who's receiving it
    -- Pick realization season (when the pick will be used)
    COALESCE(
      pick_map['season'],
      CAST(CAST(trade_season AS INT) + 1 AS STRING)
    ) AS pick_season
  FROM exploded
)
SELECT
  league_id,
  transaction_id,
  trade_season,
  round,
  original_owner_id,  -- Track the original owner (whose pick it naturally belongs to)
  from_roster_id,  -- Who traded it in THIS transaction
  to_roster_id AS side_roster_id,  -- Who received it in THIS transaction
  pick_season,
  CAST(pick_season AS INT) - CAST(trade_season AS INT) AS seasons_until_realization
FROM normalized;

In [ ]:
-- =============================================================================
-- bridge_trade_pick_to_player: Maps traded draft picks to drafted players
-- =============================================================================
-- PURPOSE:
--   Matches draft picks included in trades to the players actually drafted with
--   those picks. Enables calculation of pick value by tracking career points of
--   players drafted with traded picks.
--
-- GRAIN:
--   One row per (transaction, pick) - same pick appears multiple times if re-traded
--
-- COLUMNS:
--   league_id, transaction_id, side_roster_id, trade_season, pick_season, round,
--   original_owner_id, from_roster_id, pick_no, player_id, draft_id, 
--   drafter_roster_id, is_realized
--
-- DEPENDENCIES:
--   - fact_trade_pick_assets: Extracted traded picks from transactions
--   - stg_trade_transactions: Trade context with cluster_key
--   - dim_draft_metadata: Pre-calculated pick numbers by roster/round/draft (from core)
--   - dim_draft_picks: Actual drafted players (from core)
-- =============================================================================
CREATE OR REPLACE MATERIALIZED VIEW bridge_trade_pick_to_player AS
WITH trade_picks_with_cluster AS (
  SELECT
    pa.*,
    st.cluster_key
  FROM fact_trade_pick_assets pa
  JOIN stg_trade_transactions st ON pa.transaction_id = st.transaction_id
)
SELECT
  tp.league_id,
  tp.transaction_id,
  tp.side_roster_id,
  tp.trade_season,
  tp.pick_season,
  tp.round,
  tp.original_owner_id,
  tp.from_roster_id,
  dm.pick_no,  -- Get pre-calculated pick number from core
  dp.player_id,
  dp.draft_id,
  dp.roster_id AS drafter_roster_id,
  CASE
    WHEN dp.player_id IS NOT NULL THEN TRUE
    ELSE FALSE
  END AS is_realized
FROM trade_picks_with_cluster tp
-- Get the pick number that the original owner had for this round/season
LEFT JOIN workspace.sleeper_core.dim_draft_metadata dm
  ON tp.cluster_key = dm.cluster_key
  AND tp.pick_season = dm.season
  AND tp.original_owner_id = dm.roster_id
  AND tp.round = dm.round
-- Match to the actual drafted player using the calculated pick number
LEFT JOIN workspace.sleeper_core.dim_draft_picks dp
  ON dm.cluster_key = dp.cluster_key
  AND dm.season = dp.season
  AND dm.pick_no = dp.pick_no;

In [ ]:
-- Career points and VOR from drafted players
-- Uses cluster_key for cross-season tracking (league_id changes each season in Sleeper)
-- Counts ALL career points scored by the player regardless of subsequent ownership
CREATE OR REPLACE MATERIALIZED VIEW fact_trade_pick_points_career AS
WITH realized_picks AS (
  SELECT
    bp.*,
    st.cluster_key
  FROM bridge_trade_pick_to_player bp
  JOIN stg_trade_transactions st ON bp.transaction_id = st.transaction_id
  WHERE bp.is_realized = TRUE
),
-- Get ALL points and VOR scored by the drafted player
career_points AS (
  SELECT
    rp.league_id,
    rp.transaction_id,
    rp.side_roster_id,
    rp.trade_season,
    rp.pick_season,
    rp.player_id,
    pw.season AS scoring_season,
    pw.week AS scoring_week,
    pw.points,
    COALESCE(vor.vor, 0.0) AS vor,
    CAST(pw.season AS INT) - CAST(rp.pick_season AS INT) AS seasons_after_draft
  FROM realized_picks rp
  JOIN workspace.sleeper_core.fact_player_week_enriched pw
    ON rp.cluster_key = pw.cluster_key  -- Match on cluster (not league_id)
    AND rp.player_id = pw.player_id      -- Match on player
  LEFT JOIN workspace.sleeper_core.agg_player_week_vor vor
    ON pw.league_id = vor.league_id
    AND pw.season = vor.season
    AND pw.week = vor.week
    AND pw.player_id = vor.player_id
  WHERE
    -- Only count points after the pick was used
    CAST(pw.season AS INT) >= CAST(rp.pick_season AS INT)
)
SELECT
  league_id,
  transaction_id,
  side_roster_id,
  player_id,
  trade_season,
  pick_season,
  scoring_season,
  scoring_week,
  points,
  vor,
  seasons_after_draft
FROM career_points;

In [ ]:
-- Deduplicated draft picks view (one row per unique pick)
-- Use this when you want to count picks, not trade transactions
-- The bridge_trade_pick_to_player table has one row per trade, so the same pick
-- appears multiple times if it was traded more than once
CREATE OR REPLACE MATERIALIZED VIEW bridge_trade_pick_unique AS
WITH pick_with_cluster AS (
  SELECT
    bp.*,
    st.cluster_key,
    st.cluster_name
  FROM workspace.sleeper_trades.bridge_trade_pick_to_player bp
  JOIN workspace.sleeper_trades.stg_trade_transactions st
    ON bp.transaction_id = st.transaction_id
),
-- Get the most recent trade for each unique pick
ranked_picks AS (
  SELECT
    *,
    ROW_NUMBER() OVER (
      PARTITION BY cluster_key, pick_season, round, original_owner_id
      ORDER BY trade_season DESC, transaction_id DESC
    ) AS rn
  FROM pick_with_cluster
)
SELECT
  cluster_key,
  cluster_name,
  pick_season,
  round,
  original_owner_id,
  side_roster_id AS final_owner_id,  -- Who ended up with the pick
  transaction_id AS last_trade_id,
  player_id,
  pick_no,  -- The pick number based on original owner's slot
  draft_id,
  is_realized
FROM ranked_picks
WHERE rn = 1;